# Bidirectional Selective State Space Models (Vision Mamba & BiMamba)

## 1. Causal Language vs Non-Causal Sequences
Standard Mamba is strictly **causal**: token representations at time $t$ depend solely on past tokens $\{x_1, \dots, x_t\}$. This matches autoregressive next-token language modeling.

However, many domains are fundamentally **non-causal**:
1. **Computer Vision (ViT / Vision Mamba)**: Flattened 2D image patches have spatial relationships in all directions (top, bottom, left, right). Enforcing an artificial unidirectional raster scan limits receptive fields.
2. **Computational Genomics (DNA / RNA)**: Biological DNA strands are non-causal; regulatory motifs and gene promoters interact bidirectionally across chromatin structures.

### Bidirectional Selective SSM Architecture
Vision Mamba (Vim - Zhu et al., ICML 2024) introduces bidirectional selective scanning:
- **Forward Pass**: Scans patches from $1 \to L$, maintaining hidden state $h_t^f$.
- **Backward Pass**: Scans patches in reverse from $L \to 1$, maintaining hidden state $h_t^b$.
- **Dynamic Gating & Fusion**: Merges both directional states with linear $O(L)$ computational complexity!

$$h_t^f = \bar{A}_t^f h_{t-1}^f + \bar{B}_t^f x_t, \quad y_t^f = C_t^f h_t^f$$
$$h_t^b = \bar{A}_t^b h_{t+1}^b + \bar{B}_t^b x_t, \quad y_t^b = C_t^b h_t^b$$
$$y_t = y_t^f + y_t^b + D x_t$$

In [ ]:
import numpy as np

class SimulatedBidirectionalSSM:
    """
    Demonstrates Bidirectional Selective State Space Scanning with O(L) complexity.
    """
    def __init__(self, d_model=16, d_state=8):
        self.d_model = d_model
        self.d_state = d_state
        
        # State transitions
        self.A_f = np.eye(d_state) * 0.9
        self.A_b = np.eye(d_state) * 0.9
        self.B = np.random.randn(d_state, d_model) * 0.1
        self.C = np.random.randn(d_model, d_state) * 0.1
        self.D = np.ones(d_model) * 0.05

    def forward_scan(self, x_seq):
        """Forward causal scan: 0 -> L-1"""
        L = len(x_seq)
        h = np.zeros(self.d_state)
        y_f = np.zeros((L, self.d_model))
        for t in range(L):
            h = self.A_f @ h + self.B @ x_seq[t]
            y_f[t] = self.C @ h
        return y_f

    def backward_scan(self, x_seq):
        """Backward scan: L-1 -> 0"""
        L = len(x_seq)
        h = np.zeros(self.d_state)
        y_b = np.zeros((L, self.d_model))
        for t in reversed(range(L)):
            h = self.A_b @ h + self.B @ x_seq[t]
            y_b[t] = self.C @ h
        return y_b

    def forward(self, x_seq):
        """Full bidirectional pass with O(L) compute."""
        y_f = self.forward_scan(x_seq)
        y_b = self.backward_scan(x_seq)
        
        # Merge forward + backward + residual skip
        y = y_f + y_b + x_seq * self.D
        return y

# Verification
seq_len = 64
dim = 16
dummy_tokens = np.random.randn(seq_len, dim)

bimamba = SimulatedBidirectionalSSM(d_model=dim, d_state=8)
out_tokens = bimamba.forward(dummy_tokens)

print(f"Sequence Length : {seq_len} tokens")
print(f"Input Shape     : {dummy_tokens.shape}")
print(f"Output Shape    : {out_tokens.shape}")
print("Bidirectional receptive field achieved across all spatial tokens with O(L) complexity!")

## 2. Computational Complexity Scaling: ViT vs Vision Mamba
- **Vision Transformer (ViT)**: $O(L^2 \cdot d)$ where $L = (H/P) \times (W/P)$. For high-resolution images ($1024 \times 1024$ with $16 \times 16$ patches, $L = 4096$), quadratic attention requires massive GPU VRAM.
- **Vision Mamba (Vim)**: $O(L \cdot d)$ linear complexity, reducing memory consumption by over $75\%$ for fine-grained 1K+ image resolutions.

## 3. High-Yield Interview Questions & Best Practices

### Q1: Why can't we simply run standard bidirectional LSTM instead of Bidirectional Mamba?
**Answer:** LSTMs have sequential recurrent step dependencies that cannot be parallelized during training, making training prohibitively slow on GPUs. Mamba and S4 formulate recurrent state updates as an **associative scan (prefix sum)**, allowing parallel hardware execution across all tokens during training via associative scan kernels while maintaining $O(L)$ linear memory and fast inference.

### Q2: What is the 4-way cross-scan mechanism in VMamba?
**Answer:** In 2D images, scanning only left-to-right and right-to-left misses top-to-bottom vertical correlations. **VMamba (Visual State Space Model)** introduces a 4-way Cross-Scan Mechanism (CSM) that scans image patches along four distinct trajectories:
1. Top-Left to Bottom-Right (horizontal)
2. Bottom-Right to Top-Left (horizontal reverse)
3. Top-Left to Bottom-Right (vertical column-wise)
4. Bottom-Right to Top-Left (vertical reverse)
Ensuring every patch aggregates information from its complete 2D neighborhood.